# Traffic Sign Robustness Assessment - Solution

This notebook evaluates a traffic sign recognition model under clean, environmental, and adversarial conditions, then writes a quantitative scorecard and operational assessment report.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
os.environ["ART_DATA_PATH"] = str(ROOT / "art_data")
os.environ["USERPROFILE"] = str(ROOT)
os.environ["HOME"] = str(ROOT)
os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")

import torch
from traffic_sign_robustness_utils import (
    calculate_condition_metrics,
    environmental_test_sets,
    load_subset,
    perturbation_linf,
    plot_clean_vs_degraded,
    plot_metric_bars,
    prepare_gtsrb_subsets,
    row_from_metrics,
    run_fgsm_attacks,
    train_or_load_model,
    write_assessment_report,
    write_scorecard,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

## 1. Load the Traffic Sign Dataset and Model

In [2]:
data_dir = ROOT / "data" / "generated"
download_dir = ROOT / "data" / "gtsrb"
model_path = ROOT / "models" / "traffic_sign_cnn.pt"
results_dir = ROOT / "results"

prepare_gtsrb_subsets(data_dir, download_dir, train_per_class=80, val_per_class=20)
train_images, train_labels = load_subset(data_dir, "train")
val_images, val_labels = load_subset(data_dir, "val")

model = train_or_load_model(model_path, train_images, train_labels, device=device, epochs=12)
val_images.shape, val_labels.shape

epoch 1/12 loss=1.7304


epoch 2/12 loss=1.2505


epoch 3/12 loss=0.7715


epoch 4/12 loss=0.5202


epoch 5/12 loss=0.3469


...

epoch 9/12 loss=0.0901


epoch 10/12 loss=0.0881


epoch 11/12 loss=0.0501


epoch 12/12 loss=0.0334


((120, 3, 64, 64), (120,))

## 2. Measure Clean Baseline Metrics

In [3]:
clean_metrics = calculate_condition_metrics(model, val_images, val_labels, device=device)
clean_predictions = clean_metrics["predictions"]
clean_confidence = clean_metrics["confidence"]

rows = [row_from_metrics("clean", "baseline", clean_metrics, perturbation=0.0)]
rows

[{'condition': 'clean',
  'condition_type': 'baseline',
  'accuracy': 0.4583,
  'mean_confidence': 0.8668,
  'confidence_drop': 0.0,
  'attack_success_rate': 0.0,
  'perturbation_linf': 0.0,
  'risk_level': 'high'}]

## 3. Evaluate Environmental Perturbations

In [4]:
first_environmental_set = None

for condition, degraded_images in environmental_test_sets(val_images).items():
    metrics = calculate_condition_metrics(
        model,
        degraded_images,
        val_labels,
        clean_predictions=clean_predictions,
        clean_confidence=clean_confidence,
        device=device,
    )
    rows.append(row_from_metrics(condition, "environmental", metrics, perturbation_linf(val_images, degraded_images)))
    if first_environmental_set is None:
        first_environmental_set = (condition, degraded_images)

[row for row in rows if row["condition_type"] == "environmental"]

[{'condition': 'gaussian_noise_sigma_0.08',
  'condition_type': 'environmental',
  'accuracy': 0.4667,
  'mean_confidence': 0.8893,
  'confidence_drop': -0.0225,
  'attack_success_rate': 0.0909,
  'perturbation_linf': 0.3152,
  'risk_level': 'high'},
 {'condition': 'gaussian_blur_radius_1.0',
  'condition_type': 'environmental',
  'accuracy': 0.4583,
  'mean_confidence': 0.867,
  'confidence_drop': -0.0002,
  'attack_success_rate': 0.0,
  'perturbation_linf': 0.1664,
  'risk_level': 'high'},
 {'condition': 'jpeg_quality_35',
  'condition_type': 'environmental',
  'accuracy': 0.4583,
  'mean_confidence': 0.8617,
  'confidence_drop': 0.0051,
  'attack_success_rate': 0.0182,
  'perturbation_linf': 0.1939,
  'risk_level': 'high'}]

## 4. Run ART Adversarial Perturbations

In [5]:
first_adversarial_set = None

for condition, adversarial_images in run_fgsm_attacks(model, val_images, epsilons=[0.01, 0.03, 0.06]).items():
    metrics = calculate_condition_metrics(
        model,
        adversarial_images,
        val_labels,
        clean_predictions=clean_predictions,
        clean_confidence=clean_confidence,
        device=device,
    )
    rows.append(row_from_metrics(condition, "adversarial", metrics, perturbation_linf(val_images, adversarial_images)))
    if first_adversarial_set is None:
        first_adversarial_set = (condition, adversarial_images)

[row for row in rows if row["condition_type"] == "adversarial"]

C:\Users\joshd\OneDrive\Desktop\Udacity\Course\cd15148-ai-security-c2-demos-exercises\.module19-test-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[{'condition': 'art_fgsm_eps_0.01',
  'condition_type': 'adversarial',
  'accuracy': 0.425,
  'mean_confidence': 0.8248,
  'confidence_drop': 0.042,
  'attack_success_rate': 0.2,
  'perturbation_linf': 0.01,
  'risk_level': 'high'},
 {'condition': 'art_fgsm_eps_0.03',
  'condition_type': 'adversarial',
  'accuracy': 0.3333,
  'mean_confidence': 0.8638,
  'confidence_drop': 0.0029,
  'attack_success_rate': 0.5273,
  'perturbation_linf': 0.03,
  'risk_level': 'high'},
 {'condition': 'art_fgsm_eps_0.06',
  'condition_type': 'adversarial',
  'accuracy': 0.2333,
  'mean_confidence': 0.9273,
  'confidence_drop': -0.0605,
  'attack_success_rate': 0.8182,
  'perturbation_linf': 0.06,
  'risk_level': 'high'}]

## 5. Generate Required Outputs

In [6]:
scorecard_path = write_scorecard(rows, results_dir / "traffic_sign_robustness_scorecard.csv")
chart_path = plot_metric_bars(rows, results_dir / "traffic_sign_metric_comparison.png")
report_path = write_assessment_report(rows, results_dir / "traffic_sign_assessment_report.md")

if first_environmental_set is not None:
    condition, degraded_images = first_environmental_set
    plot_clean_vs_degraded(val_images, degraded_images, results_dir / "sample_environmental_degradation.png", f"Clean vs {condition}")

if first_adversarial_set is not None:
    condition, adversarial_images = first_adversarial_set
    plot_clean_vs_degraded(val_images, adversarial_images, results_dir / "sample_adversarial_degradation.png", f"Clean vs {condition}")

scorecard_path, chart_path, report_path

(WindowsPath('C:/Users/joshd/OneDrive/Desktop/Udacity/Course/cd15148-ai-security-c2-demos-exercises/module-19-apply-quantitative-robustness-testing/exercise/solution/results/traffic_sign_robustness_scorecard.csv'),
 WindowsPath('C:/Users/joshd/OneDrive/Desktop/Udacity/Course/cd15148-ai-security-c2-demos-exercises/module-19-apply-quantitative-robustness-testing/exercise/solution/results/traffic_sign_metric_comparison.png'),
 WindowsPath('C:/Users/joshd/OneDrive/Desktop/Udacity/Course/cd15148-ai-security-c2-demos-exercises/module-19-apply-quantitative-robustness-testing/exercise/solution/results/traffic_sign_assessment_report.md'))

In [7]:
sorted(rows, key=lambda row: (row["risk_level"], row["accuracy"]))

[{'condition': 'art_fgsm_eps_0.06',
  'condition_type': 'adversarial',
  'accuracy': 0.2333,
  'mean_confidence': 0.9273,
  'confidence_drop': -0.0605,
  'attack_success_rate': 0.8182,
  'perturbation_linf': 0.06,
  'risk_level': 'high'},
 {'condition': 'art_fgsm_eps_0.03',
  'condition_type': 'adversarial',
  'accuracy': 0.3333,
  'mean_confidence': 0.8638,
  'confidence_drop': 0.0029,
  'attack_success_rate': 0.5273,
  'perturbation_linf': 0.03,
  'risk_level': 'high'},
 {'condition': 'art_fgsm_eps_0.01',
  'condition_type': 'adversarial',
  'accuracy': 0.425,
  'mean_confidence': 0.8248,
  'confidence_drop': 0.042,
  'attack_success_rate': 0.2,
  'perturbation_linf': 0.01,
  'risk_level': 'high'},
 {'condition': 'clean',
  'condition_type': 'baseline',
  'accuracy': 0.4583,
  'mean_confidence': 0.8668,
  'confidence_drop': 0.0,
  'attack_success_rate': 0.0,
  'perturbation_linf': 0.0,
  'risk_level': 'high'},
 {'condition': 'gaussian_blur_radius_1.0',
  'condition_type': 'environmen